# BitWave 기반 Bit-Column Sparsity DNN 가속기
## Python Golden Model · 데이터 전처리 파이프라인

**과제**: 디자인프로젝트 (서강대학교 전자공학과, 2026.03–06) · 3인 팀 · Bitwave 기반 bit-column serial DNN 가속기의 FPGA 구현 및 검증

**담당 파트 (박수빈)**: MNIST MLP 학습 · int8 양자화 · sign-magnitude 변환 · Bit-flip · ZCIP 입력 데이터(ZCI, weight column) 생성 · RTL 검증용 golden 값 생성

이 노트북은 RTL(ZCIP·BCE 모듈)에 들어갈 **입력 데이터**를 만들고, RTL 출력과 비교할 **정답(golden)** 을 계산합니다.

```
MNIST MLP 학습 (FP32)
   └─▶ int8 양자화 (±127, sign-magnitude 표현 가능 범위)
         └─▶ Bit-flip (그룹 내 zero bit-column 수 늘리기)
               └─▶ ZCIP 형식 압축 (ZCI + non-zero weight column)
                     └─▶ Bit-column-serial 연산 golden (BCE와 동일 연산)
                           └─▶ testbench용 hex 파일 (Vivado $readmemh)
```

| 섹션 | 내용 |
|---|---|
| 0 | 설정 (seed, target 값) |
| 1 | 데이터 · 모델 학습 / 로드 |
| 2 | int8 양자화 · sign-magnitude 변환 |
| 3 | ZCIP 형식 압축 (ZCI) |
| 4 | Bit-flip |
| 5 | **정확도 vs zero-column 비율 trade-off** |
| 6 | 선택한 설정의 bit-flip 효과 |
| 7 | **Golden 모델 검증** (bit-column-serial 연산 == 정수 내적) |
| 8 | **하드웨어와 동일한 정수 경로로 전체 테스트셋 정확도** |
| 9 | 단일 이미지 데모 · testbench 파일 생성 |
| 10 | 결과 요약 |
| 11 | 개발 과정에서 수정한 부분 |

**실행 방법**: 위에서부터 `Run All`. 처음 실행 시 MNIST를 내려받아 학습한 뒤 weight를 `mlp_mnist_seed42.pt`로 저장하고, 이후에는 저장된 weight를 불러와 같은 결과를 재현합니다.

In [ ]:
import os
import itertools
import copy
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cpu")   # 재현성을 위해 CPU 고정 (MLP라 CPU로 충분)
print("torch", torch.__version__, "| device:", device)

## 0. 설정

`TARGET_ZERO_COLS`는 Bit-flip 후 **8개 weight 그룹마다 최소 몇 개의 magnitude bit-column을 0으로 만들지** 정하는 값입니다 (0–6).
값이 클수록 건너뛸 수 있는 column이 늘어 연산량은 줄지만 weight 왜곡이 커집니다. 5번 섹션에서 0–6 전체를 비교한 뒤 정할 수 있습니다.

k = 5이면 모든 그룹의 non-zero column이 최대 2개가 되어 zero-column 비율이 5/7 = **71.4%** 가 됩니다 (포스터 결과와 동일한 설정).

In [ ]:
SEED        = 42
GROUP_SIZE  = 8      # BCE 한 번에 처리하는 weight 수 (lane 수)
MAG_BITS    = 7      # int8 sign-magnitude의 magnitude bit 수 (b0~b6)

EPOCHS      = 3
LR          = 1e-3
TRAIN_BATCH = 64
TEST_BATCH  = 1000

# Bit-flip 강도: 그룹당 0으로 만들 magnitude column 수
TARGET_ZERO_COLS = {"fc1": 5, "fc2": 5}

DATA_DIR  = "./data"
CKPT_PATH = f"mlp_mnist_seed{SEED}.pt"
TB_DIR    = "tb_data"            # testbench 파일 출력 폴더
DEMO_INDEX = 0                   # 데모에 쓸 테스트 이미지 번호

## 1. 데이터 · 모델

784 → 120 → 10 구조의 2-layer MLP입니다. CNN·ResNet도 시도했지만, FC layer만으로 구성돼 weight를 8개씩 그룹으로 묶어 BCE에 바로 넣을 수 있고 bit-level sparsity 분석이 쉬운 MLP를 선택했습니다.

In [ ]:
set_seed(SEED)

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
test_loader  = DataLoader(test_dataset,  batch_size=TEST_BATCH,  shuffle=False)

print(f"train {len(train_dataset)} / test {len(test_dataset)}")

In [ ]:
class MLPNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 120)
        self.fc2 = nn.Linear(120, 10)

    def forward(self, x):
        x = x.reshape(x.size(0), -1)      # [B,1,28,28] -> [B,784]
        x = F.relu(self.fc1(x))
        return self.fc2(x)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for images, labels in loader:
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total

In [ ]:
model = MLPNet().to(device)

if os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    print(f"저장된 weight 로드: {CKPT_PATH}")
else:
    set_seed(SEED)
    model = MLPNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, criterion)
        acc  = evaluate(model, test_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | loss = {loss:.4f} | test acc = {acc:.2f}%")
    torch.save(model.state_dict(), CKPT_PATH)
    print(f"weight 저장: {CKPT_PATH}")

acc_float = evaluate(model, test_loader)
print(f"FP32 test accuracy: {acc_float:.2f}%")

## 2. int8 양자화 · sign-magnitude 변환

- 대칭 양자화: `scale = max|w| / 127`, `q = round(w / scale)`
- 범위를 **−127 ~ +127**로 제한하는 이유: BitWave는 sign-magnitude 표현을 쓰는데, 8bit sign-magnitude는 부호 1bit + 크기 7bit라 −128을 표현할 수 없습니다.
- sign-magnitude로 바꾸면 음수도 크기 비트가 작게 유지됩니다. 2's complement에서는 −1이 `11111111`이라 상위 bit-column이 모두 1이 되어 zero-column이 생기기 어렵습니다.

In [ ]:
def quantize_int8(x):
    """대칭 int8 양자화. x: float32 numpy 배열 -> (int16 배열, scale)"""
    x = np.asarray(x, dtype=np.float32)
    max_abs = float(np.abs(x).max())
    scale = max_abs / 127.0 if max_abs != 0 else 1.0
    q = np.clip(np.round(x / scale), -127, 127).astype(np.int16)
    return q, scale


def int_to_signmag_byte(x):
    """signed int -> 8bit sign-magnitude  (+99 -> 0x63, -99 -> 0xE3)"""
    x = int(x)
    mag = min(abs(x), 127)
    return (0x80 | mag) if x < 0 else mag


def signmag_byte_to_int(b):
    b = int(b) & 0xFF
    mag = b & 0x7F
    return -mag if (b >> 7) & 1 else mag


def pack_bits_to_int(bits):
    """bits[0]이 LSB (lane0 = LSB, RTL 입력 형식과 동일)"""
    val = 0
    for i, b in enumerate(bits):
        val |= (int(b) & 1) << i
    return val


def pack_activation_to_act64(activation):
    """activation 8개 -> 64bit sign-magnitude (lane0 = 하위 byte)"""
    assert len(activation) == GROUP_SIZE
    act64, act_sm = 0, []
    for i, a in enumerate(activation):
        sm = int_to_signmag_byte(a)
        act_sm.append(sm)
        act64 |= sm << (8 * i)
    return act64, act_sm


def to_18bit(x):  return int(x) & ((1 << 18) - 1)
def to_32bit(x):  return int(x) & 0xFFFFFFFF

def from_32bit_signed(x):
    x = int(x)
    return x - (1 << 32) if x & (1 << 31) else x

In [ ]:
# 변환 함수 단위 테스트 (경계값 포함)
for x in [0, 1, -1, 99, -99, 127, -127]:
    sm = int_to_signmag_byte(x)
    assert signmag_byte_to_int(sm) == x
    print(f"{x:5d} -> 0x{sm:02X} -> {signmag_byte_to_int(sm):5d}")

assert int_to_signmag_byte(99) == 0x63 and int_to_signmag_byte(-99) == 0xE3
assert pack_bits_to_int([1, 0, 0, 0, 0, 0, 0, 0]) == 0x01   # lane0 = LSB
assert pack_bits_to_int([0, 0, 0, 0, 0, 0, 0, 1]) == 0x80
print("sign-magnitude / bit packing 테스트 통과")

## 3. ZCIP 형식 압축

weight를 출력 뉴런마다 8개씩 그룹으로 묶고, 그룹 안에서 **같은 bit 위치끼리 column**으로 봅니다. 그룹의 8개 weight가 모두 0인 bit-column은 BCE가 연산할 필요가 없으니 저장·연산에서 뺍니다.

| 출력 | 형식 | 의미 |
|---|---|---|
| `zci` (8bit) | `zci[shift]` (shift = 0–6) | magnitude bit `shift`의 column이 0이 아니면 1 |
|  | `zci[7]` | 그룹에 음수 weight가 있으면 1 (sign column 존재) |
| `weight_col` (8bit) | lane0 = LSB | non-zero column만 **shift 0 → 6 순서**로 저장 |

ZCIP는 `zci`를 보고 non-zero column 위치(=shift 양)와 연산할 cycle 수를 정합니다.

In [ ]:
def compress_zcip(q_weight, group_size=GROUP_SIZE):
    """[out, in] int 배열 -> 뉴런·그룹별 ZCI와 non-zero magnitude column"""
    q_weight = np.asarray(q_weight)
    out_f, in_f = q_weight.shape
    result = {"out_features": out_f, "in_features": in_f, "group_size": group_size,
              "groups_per_neuron": (in_f + group_size - 1) // group_size, "neurons": []}

    for n in range(out_f):
        groups = []
        for gidx, start in enumerate(range(0, in_f, group_size)):
            end = min(start + group_size, in_f)
            w = [int(x) for x in q_weight[n, start:end]]
            w += [0] * (group_size - len(w))                 # 마지막 그룹 0 padding

            mags  = [min(abs(x), 127) for x in w]
            signs = [1 if x < 0 else 0 for x in w]

            zci = (1 << 7) if any(signs) else 0
            mag_cols = []
            for shift in range(MAG_BITS):                    # shift 0 -> 6 순서
                bits = [(m >> shift) & 1 for m in mags]
                if any(bits):
                    zci |= 1 << shift
                    mag_cols.append({"shift": shift, "bits": bits,
                                     "packed": pack_bits_to_int(bits)})

            groups.append({"neuron_index": n, "group_index": gidx,
                           "input_range": (start, end - 1), "weights_int8": w,
                           "sign_vals": signs, "zci_packed": zci, "mag_cols": mag_cols})
        result["neurons"].append({"neuron_index": n, "groups": groups})
    return result


def nonzero_cols_per_group(q_weight, group_size=GROUP_SIZE):
    """그룹별 non-zero magnitude column 수 [out, G] (빠른 numpy 버전)"""
    q = np.asarray(q_weight)
    out_f, in_f = q.shape
    assert in_f % group_size == 0
    mags = np.abs(q).reshape(out_f, in_f // group_size, group_size)
    bits = (mags[..., None] >> np.arange(MAG_BITS)) & 1      # [out, G, lane, bit]
    return bits.any(axis=2).sum(axis=-1)                     # [out, G]


def zero_col_ratio(q_weight):
    nz = nonzero_cols_per_group(q_weight)
    return 100.0 * (MAG_BITS * nz.size - nz.sum()) / (MAG_BITS * nz.size)

## 4. Bit-flip

그룹 단위로 **0으로 만들 magnitude column 조합**(7개 중 k개, 7Ck가지)을 모두 시도하고, 그 조합에서 허용되는 값 중 원래 값과 가장 가까운 값으로 각 weight를 바꿉니다. 원래 그룹과의 **제곱오차가 가장 작은 조합**을 선택합니다.

- 부호는 바꾸지 않고 크기(magnitude)만 조정합니다.
- 이미 zero column이 k개 이상인 그룹은 오차 0인 조합이 있어 그대로 유지됩니다.
- 논문의 bit-flip은 layer별로 정확도를 확인하며 그룹 크기와 zero column 수를 greedy하게 늘려가는 방식입니다. 여기서는 target k를 고정하고 5번 섹션에서 k별 정확도를 비교하는 방식으로 단순화했습니다.
- 후보 값을 조합별로 미리 계산하고 numpy로 전체 행렬을 한 번에 처리합니다 (초기 버전의 그룹별 3중 for문 대비 대폭 단축).

In [ ]:
def _precompute_candidates(target_zero_cols):
    """조합별로 '해당 bit들이 0인' magnitude 후보값 목록"""
    all_mags = np.arange(128, dtype=np.int16)
    combos = {}
    for zero_cols in itertools.combinations(range(MAG_BITS), target_zero_cols):
        mask = np.ones(128, dtype=bool)
        for col in zero_cols:
            mask &= ((all_mags >> col) & 1) == 0
        combos[zero_cols] = all_mags[mask]
    return combos


def bitflip_q_weight(q_weight, target_zero_cols, group_size=GROUP_SIZE):
    """[out, in] int 배열 -> 그룹마다 zero magnitude column이 target개 이상인 배열"""
    q = np.asarray(q_weight).astype(np.int16)
    out_f, in_f = q.shape
    assert in_f % group_size == 0, "in_features는 group_size의 배수여야 함"
    G = in_f // group_size

    signs  = np.where(q < 0, -1, 1).astype(np.int16)
    mags_g = np.abs(q).reshape(out_f, G, group_size)

    best      = mags_g.copy()
    best_dist = np.full((out_f, G), np.inf)

    for cand in _precompute_candidates(target_zero_cols).values():
        idx     = np.abs(cand[None, None, None, :] - mags_g[..., None]).argmin(axis=-1)
        nearest = cand[idx]                                          # [out, G, lane]
        dist    = ((nearest - mags_g).astype(np.int32) ** 2).sum(-1) # [out, G]
        better  = dist < best_dist
        best[better]      = nearest[better]
        best_dist[better] = dist[better]

    return (signs * best.reshape(out_f, in_f)).astype(np.int16)

In [ ]:
def plot_group_bits(before, after, title=""):
    """한 그룹(8 weight)의 magnitude bit 행렬을 bit-flip 전후로 비교"""
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, vals, name in zip(axes, [before, after], ["Before bit-flip", "After bit-flip"]):
        mags = np.abs(np.asarray(vals))
        mat  = np.array([[(m >> b) & 1 for b in range(MAG_BITS - 1, -1, -1)] for m in mags])
        for c in range(MAG_BITS):
            if mat[:, c].sum() == 0:                      # zero column 강조
                ax.axvspan(c - 0.5, c + 0.5, color="#f2c94c", alpha=0.35, lw=0)
        ax.imshow(np.ma.masked_where(mat == 0, mat), cmap="Greys", vmin=0, vmax=1.4,
                  aspect="equal")
        ax.set_xticks(range(MAG_BITS))
        ax.set_xticklabels([f"b{b}" for b in range(MAG_BITS - 1, -1, -1)])
        ax.set_yticks(range(len(mags)))
        ax.set_yticklabels([f"{int(v):+d}" for v in vals])
        ax.set_xticks(np.arange(-0.5, MAG_BITS), minor=True)
        ax.set_yticks(np.arange(-0.5, len(mags)), minor=True)
        ax.grid(which="minor", color="#cccccc", lw=0.6)
        ax.tick_params(which="minor", length=0)
        nz = int((mat.sum(0) > 0).sum())
        ax.set_title(f"{name}\nnon-zero columns = {nz} / {MAG_BITS}")
    fig.suptitle(title + "   (yellow = zero column, skipped by ZCIP)", fontsize=10)
    plt.tight_layout()
    plt.show()

## 5. 정확도 vs zero-column 비율 trade-off

FC1·FC2에 같은 target(0–6)을 적용해 비교합니다.

- **정확도**: 양자화(+bit-flip)된 weight를 FP32 모델에 다시 넣어 테스트셋 10,000장으로 측정 (bias는 FP32 유지)
- **zero-column 비율**: 전체 magnitude bit-column 중 ZCIP가 건너뛸 수 있는 column 비율

In [ ]:
fc1_q, fc1_scale = quantize_int8(model.fc1.weight.detach().cpu().numpy())
fc2_q, fc2_scale = quantize_int8(model.fc2.weight.detach().cpu().numpy())
print("fc1:", fc1_q.shape, f"scale={fc1_scale:.6f}")
print("fc2:", fc2_q.shape, f"scale={fc2_scale:.6f}")


def accuracy_with_qweights(q1, q2):
    m = copy.deepcopy(model)
    with torch.no_grad():
        m.fc1.weight.copy_(torch.from_numpy(q1.astype(np.float32) * fc1_scale))
        m.fc2.weight.copy_(torch.from_numpy(q2.astype(np.float32) * fc2_scale))
    return evaluate(m, test_loader)


acc_quant = accuracy_with_qweights(fc1_q, fc2_q)
print(f"\nFP32 {acc_float:.2f}%  ->  int8 {acc_quant:.2f}%")

In [ ]:
sweep = []
for k in range(MAG_BITS):
    q1 = bitflip_q_weight(fc1_q, k)
    q2 = bitflip_q_weight(fc2_q, k)
    sweep.append({"k": k, "acc": accuracy_with_qweights(q1, q2),
                  "fc1_zero": zero_col_ratio(q1), "fc2_zero": zero_col_ratio(q2)})

print(f"{'target k':>8} | {'accuracy':>9} | {'drop':>6} | {'FC1 zero col':>12} | {'FC2 zero col':>12}")
print("-" * 62)
for r in sweep:
    mark = "  <- 선택" if r["k"] == TARGET_ZERO_COLS["fc1"] else ""
    print(f"{r['k']:>8} | {r['acc']:>8.2f}% | {acc_quant - r['acc']:>6.2f} | "
          f"{r['fc1_zero']:>11.1f}% | {r['fc2_zero']:>11.1f}%{mark}")

In [ ]:
ks = [r["k"] for r in sweep]
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(ks, [r["acc"] for r in sweep], "o-", color="#1f4e79", label="Accuracy")
ax1.axhline(acc_quant, color="#1f4e79", ls=":", lw=1)
ax1.set_xlabel("Target zero columns per group (k)")
ax1.set_ylabel("Test accuracy (%)", color="#1f4e79")

ax2 = ax1.twinx()
ax2.plot(ks, [r["fc1_zero"] for r in sweep], "s--", color="#c0504d", label="FC1 zero-col ratio")
ax2.plot(ks, [r["fc2_zero"] for r in sweep], "^--", color="#e3a33b", label="FC2 zero-col ratio")
ax2.set_ylabel("Zero bit-column ratio (%)")
ax2.set_ylim(0, 100)

lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc="lower left", fontsize=9)
ax1.set_title("Accuracy vs. bit-column sparsity (Bit-flip target k)")
plt.tight_layout()
plt.show()

## 6. 선택한 설정의 bit-flip 효과

0번 섹션의 `TARGET_ZERO_COLS`로 bit-flip한 weight를 이후 모든 단계(golden, testbench 파일)에 사용합니다.

**연산 cycle 추정 모델** (Python 상의 추정, RTL 측정값과는 별개)
- BCE는 그룹당 non-zero magnitude column 하나를 1 cycle에 처리, 부호는 같은 cycle에 XOR로 처리한다고 가정
- Baseline (column skip 없는 bit-serial) = 그룹당 7 cycle
- 메모리 접근·제어 오버헤드는 포함하지 않음

In [ ]:
fc1_q_bf = bitflip_q_weight(fc1_q, TARGET_ZERO_COLS["fc1"])
fc2_q_bf = bitflip_q_weight(fc2_q, TARGET_ZERO_COLS["fc2"])
acc_bitflip = accuracy_with_qweights(fc1_q_bf, fc2_q_bf)

result_fc1    = compress_zcip(fc1_q)
result_fc2    = compress_zcip(fc2_q)
result_fc1_bf = compress_zcip(fc1_q_bf)
result_fc2_bf = compress_zcip(fc2_q_bf)

print(f"TARGET_ZERO_COLS = {TARGET_ZERO_COLS}\n")
print(f"  FP32     정확도 : {acc_float:.2f}%")
print(f"  int8     정확도 : {acc_quant:.2f}%   (drop {acc_float - acc_quant:+.2f})")
print(f"  Bit-flip 정확도 : {acc_bitflip:.2f}%   (drop {acc_quant - acc_bitflip:+.2f})\n")

stats = {}
print(f"{'':6} | {'전체 col':>8} | {'zero col (전)':>13} | {'zero col (후)':>13} | {'skip 비율 전 -> 후':>18}")
print("-" * 72)
for name, qb, qa in [("FC1", fc1_q, fc1_q_bf), ("FC2", fc2_q, fc2_q_bf)]:
    nz_b, nz_a = nonzero_cols_per_group(qb), nonzero_cols_per_group(qa)
    total = MAG_BITS * nz_b.size
    zb, za = total - nz_b.sum(), total - nz_a.sum()
    stats[name] = {"groups": nz_b.size, "total": total, "nz_before": int(nz_b.sum()),
                   "nz_after": int(nz_a.sum()), "nz_b": nz_b, "nz_a": nz_a}
    print(f"{name:6} | {total:>8,} | {zb:>13,} | {za:>13,} | "
          f"{100*zb/total:>7.1f}% -> {100*za/total:5.1f}%")

In [ ]:
print("그룹별 non-zero column 수 분포")
for name in ["FC1", "FC2"]:
    db, da = Counter(stats[name]["nz_b"].ravel()), Counter(stats[name]["nz_a"].ravel())
    print(f"\n[{name}] {'NZ cols':>7} | {'Before':>7} | {'After':>7}")
    for k in range(MAG_BITS + 1):
        if db.get(k, 0) or da.get(k, 0):
            print(f"      {k:>7} | {db.get(k, 0):>7,} | {da.get(k, 0):>7,}")
    print(f"      {'평균':>6} | {stats[name]['nz_b'].mean():>7.2f} | {stats[name]['nz_a'].mean():>7.2f}")

print("\nBit-flip으로 인한 weight 왜곡 (int8 단위)")
for name, qb, qa in [("FC1", fc1_q, fc1_q_bf), ("FC2", fc2_q, fc2_q_bf)]:
    d = np.abs(qb.astype(np.int32) - qa)
    changed = d > 0
    mean_d = d[changed].mean() if changed.any() else 0.0
    print(f"  {name}: 변경 {changed.sum():,}/{d.size:,} ({100*changed.mean():.1f}%), "
          f"평균 변화 {mean_d:.2f}, 최대 변화 {d.max()}")

print("\n연산 cycle 추정 (BCE: non-zero column 1개 = 1 cycle)")
cyc = {}
for name in ["FC1", "FC2"]:
    s = stats[name]
    base = MAG_BITS * s["groups"]
    cyc[name] = (base, s["nz_before"], s["nz_after"])
    print(f"  {name}: baseline {base:,} -> column skip {s['nz_before']:,} "
          f"({100*(1-s['nz_before']/base):.1f}% 감소) -> skip + bit-flip {s['nz_after']:,} "
          f"({100*(1-s['nz_after']/base):.1f}% 감소)")
tb = sum(v[0] for v in cyc.values()); tn = sum(v[1] for v in cyc.values()); ta = sum(v[2] for v in cyc.values())
print(f"  전체: column skip만 {tb/tn:.2f}x, skip + bit-flip {tb/ta:.2f}x (baseline 대비 cycle 기준)")

In [ ]:
# bit-flip으로 zero column이 늘어난 그룹 하나를 시각화
nz_b, nz_a = stats["FC1"]["nz_b"], stats["FC1"]["nz_a"]
cand = np.argwhere(nz_a < nz_b)
assert len(cand) > 0, "bit-flip으로 바뀐 그룹이 없음 (TARGET_ZERO_COLS가 0이거나 작음)"
n, g = cand[len(cand) // 2]
s = g * GROUP_SIZE
before, after = fc1_q[n, s:s + GROUP_SIZE], fc1_q_bf[n, s:s + GROUP_SIZE]
print(f"FC1 neuron {n}, group {g}")
print("before:", before.tolist())
print("after :", after.tolist())
print("diff  :", (after - before).tolist())
plot_group_bits(before, after, title=f"FC1 neuron {n}, group {g}")

## 7. Golden 모델 검증

`calculate_group_golden`은 BCE와 **같은 순서**로 연산합니다.

1. non-zero column마다, 해당 column bit가 1인 lane의 activation magnitude를 더함 (부호 `S = A_sign XOR W_sign`)
2. column 합을 `shift`만큼 왼쪽 shift (add-then-shift)
3. 모든 column 결과를 누적 → 그룹 partial sum (18bit), 뉴런 합 (32bit)

이 결과가 **일반 정수 내적과 정확히 같은지** 자동으로 확인합니다. 같다면 testbench의 golden 값과 RTL 출력이 다를 때 원인은 RTL 쪽(데이터 전달, bit 순서, 제어)으로 좁혀집니다.

**18bit인 이유**: 그룹 partial sum의 최대 크기는 8 × 127 × 127 = 129,032 < 2¹⁷ = 131,072 → 부호 포함 18bit면 overflow가 없습니다.

In [ ]:
def calculate_group_golden(g, activation):
    """한 그룹의 BCE partial sum (bit-column-serial, add-then-shift)"""
    act64, act_sm = pack_activation_to_act64(activation)
    acc, columns = 0, []
    for col in g["mag_cols"]:
        W, shift = col["packed"], col["shift"]
        column_sum, lanes = 0, []
        for lane in range(GROUP_SIZE):
            if not (W >> lane) & 1:
                continue
            a_sign, a_mag = (act_sm[lane] >> 7) & 1, act_sm[lane] & 0x7F
            prod_sign = a_sign ^ g["sign_vals"][lane]          # S_bce = A_sign ^ W_sign
            contribution = -a_mag if prod_sign else a_mag
            column_sum += contribution
            lanes.append({"lane": lane, "A": activation[lane], "A_sm": f"0x{act_sm[lane]:02X}",
                          "w_sign": g["sign_vals"][lane], "prod_sign": prod_sign,
                          "contribution": contribution})
        shifted = column_sum << shift
        acc += shifted
        columns.append({"shift": shift, "W": W, "W_hex": f"0x{W:02X}",
                        "column_sum": column_sum, "shifted": shifted, "lane_details": lanes})
    return {"zci": g["zci_packed"], "act64": act64, "act_sm": act_sm,
            "golden_signed": acc, "golden_18b": to_18bit(acc), "columns": columns}


def neuron_outputs_bcs(result, q_act):
    """압축된 weight로 bit-column-serial 연산 -> 뉴런별 출력, 그룹 partial sum 최대 크기"""
    outs, max_group = [], 0
    for neuron in result["neurons"]:
        total = 0
        for g in neuron["groups"]:
            s, e = g["input_range"]
            act = [int(x) for x in q_act[s:e + 1]]
            act += [0] * (GROUP_SIZE - len(act))
            gs = calculate_group_golden(g, act)["golden_signed"]
            max_group = max(max_group, abs(gs))
            total += gs
        outs.append(total)
    return np.array(outs, dtype=np.int64), max_group

In [ ]:
# (1) 임의 weight/activation 그룹 테스트: 음수 activation까지 포함해 부호 XOR 경로 전체 검증
rng = np.random.default_rng(0)
W_rand = rng.integers(-127, 128, size=(2000, GROUP_SIZE))
A_rand = rng.integers(-127, 128, size=(2000, GROUP_SIZE))
edge = np.array([[127] * 8, [-127] * 8, [0] * 8, [1, -1, 2, -2, 64, -64, 127, -127]])
W_rand = np.vstack([W_rand, edge, edge, edge[::-1]])
A_rand = np.vstack([A_rand, edge, edge[::-1], edge])

res_rand = compress_zcip(W_rand)
mismatch = 0
for i, neuron in enumerate(res_rand["neurons"]):
    got = calculate_group_golden(neuron["groups"][0], A_rand[i].tolist())["golden_signed"]
    ref = int(np.dot(W_rand[i].astype(np.int64), A_rand[i].astype(np.int64)))
    mismatch += (got != ref)
    assert abs(got) < 2 ** 17, "18bit 범위 초과"

print(f"임의 그룹 {len(W_rand):,}개: bit-column-serial 결과 vs 정수 내적 불일치 {mismatch}개")
assert mismatch == 0
print("음수 activation 포함 부호 처리 검증 통과")

In [ ]:
# (2) 실제 학습된 weight + 테스트 이미지 1장으로 FC1, FC2 전체 뉴런 검증
img_t, label = test_dataset[DEMO_INDEX]
img_flat = img_t.reshape(-1).numpy()

q_act1, act1_scale = quantize_int8(img_flat)
fc1_bcs, max_g1 = neuron_outputs_bcs(result_fc1_bf, q_act1)
fc1_ref = fc1_q_bf.astype(np.int64) @ q_act1.astype(np.int64)

q_act2, act2_scale = quantize_int8(np.maximum(fc1_ref, 0).astype(np.float32))   # ReLU -> 재양자화
fc2_bcs, max_g2 = neuron_outputs_bcs(result_fc2_bf, q_act2)
fc2_ref = fc2_q_bf.astype(np.int64) @ q_act2.astype(np.int64)

print(f"FC1 120개 뉴런 일치: {np.array_equal(fc1_bcs, fc1_ref)} | 그룹 partial sum 최대 |값| = {max_g1:,}")
print(f"FC2  10개 뉴런 일치: {np.array_equal(fc2_bcs, fc2_ref)} | 그룹 partial sum 최대 |값| = {max_g2:,}")
assert np.array_equal(fc1_bcs, fc1_ref) and np.array_equal(fc2_bcs, fc2_ref)
assert max(max_g1, max_g2) < 2 ** 17
print("실제 weight 기준 golden 검증 통과 (18bit 범위 이내)")

## 8. 하드웨어와 동일한 정수 경로로 테스트셋 정확도

5–6번 섹션의 정확도는 양자화된 weight를 FP32 모델에 다시 넣은 값입니다. 실제 가속기는 **정수 연산만** 하고 **bias 연산이 없으므로**, RTL과 같은 경로로 10,000장 전체 정확도를 따로 측정합니다.

```
image -> int8 양자화 -> FC1 (정수 내적) -> ReLU -> int8 재양자화 -> FC2 (정수 내적) -> argmax
```

7번 섹션에서 bit-column-serial 연산과 정수 내적이 같음을 확인했으므로, 여기서는 속도를 위해 정수 내적으로 계산합니다.

In [ ]:
def int_forward(img_flat, q1, q2):
    """RTL과 같은 정수 연산 경로 (bias 없음)"""
    qa1, _ = quantize_int8(img_flat)
    h = q1.astype(np.int64) @ qa1.astype(np.int64)
    qa2, _ = quantize_int8(np.maximum(h, 0).astype(np.float32))
    return q2.astype(np.int64) @ qa2.astype(np.int64)

In [ ]:
def int_path_accuracy(q1, q2):
    correct = total = 0
    for images, labels in test_loader:
        x = images.reshape(images.size(0), -1).numpy()
        for xi, yi in zip(x, labels.numpy()):
            correct += int(np.argmax(int_forward(xi, q1, q2)) == yi)
            total += 1
    return 100.0 * correct / total

acc_int_quant   = int_path_accuracy(fc1_q,    fc2_q)
acc_int_bitflip = int_path_accuracy(fc1_q_bf, fc2_q_bf)

print(f"{'':28} | {'FP32 모델 경로':>14} | {'정수 경로 (HW 동일)':>18}")
print("-" * 68)
print(f"{'int8 (bit-flip 전)':28} | {acc_quant:>13.2f}% | {acc_int_quant:>17.2f}%")
print(f"{'bit-flip 후 (k=' + str(TARGET_ZERO_COLS['fc1']) + ')':28} | {acc_bitflip:>13.2f}% | {acc_int_bitflip:>17.2f}%")

## 9. 단일 이미지 데모 · testbench 파일 생성

테스트 이미지 한 장에 대해 FC1 → FC2 golden 파일을 만들고, 파일에서 다시 읽은 FC2 출력으로 숫자를 예측합니다.

| 파일 | 한 줄 형식 | 내용 |
|---|---|---|
| `zci.hex` | 2 hex | 그룹별 ZCI |
| `weight_col.hex` | 2 hex | non-zero magnitude column (shift 0→6, lane0 = LSB) |
| `weight.hex` | 16 hex | 그룹 8개 weight를 sign-magnitude로 패킹 (lane0 = 하위 byte) |
| `activation.hex` | 16 hex | 그룹 8개 activation (sign-magnitude) |
| `golden.hex` | 5 hex | 그룹 partial sum (18bit 2's complement) |
| `golden_total.hex` | 8 hex | 뉴런 출력 (32bit 2's complement) |
| `info.txt`, `debug.txt` | - | 그룹별 `wcol_start`/`wcol_num`, lane별 상세 연산 (디버깅용) |

In [ ]:
plt.figure(figsize=(2.6, 2.6))
plt.imshow(img_t.squeeze().numpy(), cmap="gray")
plt.title(f"test[{DEMO_INDEX}]  label = {label}")
plt.axis("off")
plt.show()

In [ ]:
def dump_tb_data(result, q_act, out_dir, neuron_indices=None):
    """testbench용 hex 파일 생성. neuron_indices=None이면 전체 뉴런."""
    os.makedirs(out_dir, exist_ok=True)
    neurons = result["neurons"]
    if neuron_indices is not None:
        neurons = [neurons[i] for i in neuron_indices]

    total_wcols = 0
    names = ["zci.hex", "weight_col.hex", "weight.hex", "activation.hex",
             "golden.hex", "golden_total.hex", "info.txt", "debug.txt"]
    f = {n: open(os.path.join(out_dir, n), "w") for n in names}
    try:
        for neuron in neurons:
            neuron_total = 0
            for g in neuron["groups"]:
                s, e = g["input_range"]
                act = [int(x) for x in q_act[s:e + 1]]
                act += [0] * (GROUP_SIZE - len(act))
                gd = calculate_group_golden(g, act)

                w64 = 0
                for lane, w in enumerate(g["weights_int8"]):
                    w64 |= int_to_signmag_byte(w) << (8 * lane)

                f["zci.hex"].write(f"{gd['zci']:02x}\n")
                f["weight.hex"].write(f"{w64:016x}\n")
                f["activation.hex"].write(f"{gd['act64']:016x}\n")
                f["golden.hex"].write(f"{gd['golden_18b']:05x}\n")
                for col in g["mag_cols"]:
                    f["weight_col.hex"].write(f"{col['packed']:02x}\n")

                f["info.txt"].write(
                    f"neuron={neuron['neuron_index']:04d} group={g['group_index']:04d} "
                    f"input_range={s}-{e} zci=0x{gd['zci']:02X} wcol_start={total_wcols} "
                    f"wcol_num={len(g['mag_cols'])} golden=0x{gd['golden_18b']:05X} "
                    f"golden_signed={gd['golden_signed']}\n")

                d = f["debug.txt"]
                d.write(f"neuron {neuron['neuron_index']}, group {g['group_index']}\n")
                d.write(f"input_range   = {s}-{e}\n")
                d.write(f"zci           = 0x{gd['zci']:02X}\n")
                d.write(f"weight        = 0x{w64:016X}\n")
                d.write(f"activation    = {act}\n")
                d.write(f"act64         = 0x{gd['act64']:016X}\n")
                d.write(f"wcol_num      = {len(g['mag_cols'])}\n")
                for col in gd["columns"]:
                    d.write(f"  shift={col['shift']}, W={col['W_hex']}, "
                            f"column_sum={col['column_sum']}, shifted={col['shifted']}\n")
                    for ln in col["lane_details"]:
                        d.write(f"    lane{ln['lane']}: A={ln['A']}, A_sm={ln['A_sm']}, "
                                f"w_sign={ln['w_sign']}, prod_sign={ln['prod_sign']}, "
                                f"contribution={ln['contribution']}\n")
                d.write(f"golden_signed = {gd['golden_signed']}\n")
                d.write(f"golden_18b    = 0x{gd['golden_18b']:05X}\n\n")

                total_wcols  += len(g["mag_cols"])
                neuron_total += gd["golden_signed"]
            f["golden_total.hex"].write(f"{to_32bit(neuron_total):08x}\n")
    finally:
        for fh in f.values():
            fh.close()
    print(f"{out_dir}: 뉴런 {len(neurons)}개, 그룹 {len(neurons) * result['groups_per_neuron']:,}개, "
          f"weight column {total_wcols:,}개")
    return total_wcols


def read_golden_total(path):
    with open(path) as fh:
        return np.array([from_32bit_signed(int(line, 16)) for line in fh if line.strip()],
                        dtype=np.int64)

In [ ]:
# FC1 golden 생성 -> 파일에서 읽어 ReLU + 재양자화 -> FC2 입력
fc1_dir, fc2_dir = os.path.join(TB_DIR, "fc1"), os.path.join(TB_DIR, "fc2")
dump_tb_data(result_fc1_bf, q_act1, fc1_dir)

fc1_from_file = read_golden_total(os.path.join(fc1_dir, "golden_total.hex"))
q_act2_file, _ = quantize_int8(np.maximum(fc1_from_file, 0).astype(np.float32))
dump_tb_data(result_fc2_bf, q_act2_file, fc2_dir)

logits = read_golden_total(os.path.join(fc2_dir, "golden_total.hex"))
pred = int(np.argmax(logits))

print("\n── FC2 golden 출력 (logits) ──")
for i, v in enumerate(logits):
    print(f"  숫자 {i}: {v:10d}{'  <- 최대' if i == pred else ''}")
print(f"\n예측 {pred} / 정답 {label} -> {'성공' if pred == label else '실패'}")

# 파일 경로 결과가 정수 경로(int_forward)와 같은지 확인
assert np.array_equal(logits, int_forward(img_flat, fc1_q_bf, fc2_q_bf))
print("testbench 파일 경로 결과 == 정수 경로 결과 (일치)")

## 10. 결과 요약

In [ ]:
k = TARGET_ZERO_COLS
print("=" * 60)
print(f"  BitWave golden model 요약   (seed {SEED}, bit-flip k = {k['fc1']}/{k['fc2']})")
print("=" * 60)
print(f"  FP32 정확도                    : {acc_float:.2f}%")
print(f"  int8 정확도                    : {acc_quant:.2f}%")
print(f"  bit-flip 후 정확도             : {acc_bitflip:.2f}%")
print(f"  bit-flip 후 정확도 (정수 경로) : {acc_int_bitflip:.2f}%")
print("-" * 60)
for name in ["FC1", "FC2"]:
    s = stats[name]
    print(f"  {name} zero-column 비율 : {100*(1-s['nz_before']/s['total']):5.1f}% -> "
          f"{100*(1-s['nz_after']/s['total']):5.1f}%")
print(f"  연산 cycle (추정)        : baseline 대비 {tb/ta:.2f}x 감소 "
      f"(column skip만: {tb/tn:.2f}x)")
print("-" * 60)
print(f"  golden 검증 : 임의 그룹 {len(W_rand):,}개 + 실제 FC1/FC2 전체 뉴런 일치")
print(f"  testbench   : {TB_DIR}/fc1, {TB_DIR}/fc2")
print("=" * 60)

## 11. 개발 과정에서 수정한 부분

초기 버전 코드와 비교해 RTL과의 정합성, 검증 신뢰도를 위해 바꾼 내용입니다.

| 문제 | 초기 버전 | 수정 후 |
|---|---|---|
| **bit 배열 순서** | column을 MSB 먼저(b6→b0) 패킹, ZCI도 `[sign, b6, …, b0]` 순서 | RTL 입력 형식에 맞춰 lane0 = LSB, column은 shift 0→6 순서, `zci[shift]`, `zci[7] = sign` |
| **부호 처리** | `sign_col.hex`를 별도 파일로 생성 | weight를 sign-magnitude 64bit로 패킹하고, golden 계산을 BCE의 `S = A_sign XOR W_sign`과 같은 방식으로 맞춤 |
| **bit-flip 정확도 측정** | bit-flip된 weight를 모델에 넣지 않은 채 측정 → bit-flip 정확도가 FP32와 같게 나옴 | 양자화·bit-flip weight를 각각 모델에 넣어 측정 |
| **bit-flip 속도** | 그룹마다 Python 3중 for문 | 후보값 미리 계산 + numpy로 행렬 전체 한 번에 처리 |
| **재현성** | 실행마다 정확도가 달라짐 (seed 미고정) | seed 고정, 학습 weight 저장·로드 |
| **검증 범위** | 이미지 1장의 예측 결과만 확인 | bit-column-serial 결과 == 정수 내적 자동 검증(음수 activation 포함), 정수 경로로 테스트셋 10,000장 정확도 |